# Model Compatibility Check for Flask and CoreML

This notebook demonstrates how to check and ensure that your trained model is compatible with Flask applications and Apple's CoreML format without implementing the full deployment pipeline.

## Overview

When developing AI models, it's important to ensure they can be easily deployed in various environments. This notebook focuses on two common deployment targets:

1. **Flask Applications**: Web services that serve model predictions via HTTP APIs
2. **Apple's CoreML**: Framework for running machine learning models on Apple devices

We'll check model compatibility, identify potential issues, and make necessary adjustments to ensure smooth deployment.

In [ ]:
import os
import sys
import torch
import yaml
import logging
from pathlib import Path

# Add the project root to the path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

# Import project modules
from src.model.architecture import TransformerModel
from src.utils.compatibility import (
    check_flask_compatibility,
    check_coreml_compatibility,
    prepare_model_for_compatibility,
    trace_model_for_export,
    get_compatibility_report
)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 1. Load Configuration and Model

First, let's load the configuration and model.

In [ ]:
# Load configuration
config_path = project_root / 'configs' / 'config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract model configuration
model_config = config['model']

# Create output directory
output_dir = project_root / 'outputs' / 'compatibility'
os.makedirs(output_dir, exist_ok=True)

print(f"Model configuration:\n{model_config}")

In [ ]:
def load_or_create_model(model_config, checkpoint_path=None):
    """Load a model from checkpoint or create a new one."""
    # Initialize model
    model = TransformerModel(
        vocab_size=model_config['vocab_size'],
        hidden_size=model_config['hidden_size'],
        num_hidden_layers=model_config['num_hidden_layers'],
        num_attention_heads=model_config['num_attention_heads'],
        intermediate_size=model_config['intermediate_size'],
        hidden_dropout_prob=model_config.get('hidden_dropout_prob', 0.1),
        attention_probs_dropout_prob=model_config.get('attention_probs_dropout_prob', 0.1),
        max_position_embeddings=model_config['max_position_embeddings'],
        initializer_range=model_config.get('initializer_range', 0.02)
    )
    
    # Load checkpoint if provided
    if checkpoint_path and os.path.exists(checkpoint_path):
        logger.info(f"Loading model from {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        model.load_state_dict(checkpoint['model_state_dict'])
    
    return model

# Path to the model checkpoint
checkpoint_path = project_root / 'outputs' / 'checkpoints' / 'model_final.pt'

# Load or create model
model = load_or_create_model(model_config, checkpoint_path if os.path.exists(checkpoint_path) else None)

print(f"Model loaded successfully")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Check Flask Compatibility

Now, let's check if the model is compatible with Flask applications.

In [ ]:
# Check Flask compatibility
flask_compatibility = check_flask_compatibility(model)

print("Flask Compatibility Check:")
print(f"Compatible: {flask_compatibility['is_compatible']}")

if flask_compatibility['issues']:
    print("\nIssues:")
    for issue in flask_compatibility['issues']:
        print(f"- {issue}")

if flask_compatibility['recommendations']:
    print("\nRecommendations:")
    for rec in flask_compatibility['recommendations']:
        print(f"- {rec}")

## 3. Check CoreML Compatibility

Next, let's check if the model is compatible with Apple's CoreML format.

In [ ]:
# Check CoreML compatibility
coreml_compatibility = check_coreml_compatibility(model)

print("CoreML Compatibility Check:")
print(f"Compatible: {coreml_compatibility['is_compatible']}")

if coreml_compatibility['issues']:
    print("\nIssues:")
    for issue in coreml_compatibility['issues']:
        print(f"- {issue}")

if coreml_compatibility['recommendations']:
    print("\nRecommendations:")
    for rec in coreml_compatibility['recommendations']:
        print(f"- {rec}")

## 4. Prepare Model for Compatibility

Now, let's prepare the model to ensure compatibility with both Flask and CoreML.

In [ ]:
# Prepare model for compatibility
prepared_model, compatibility_info = prepare_model_for_compatibility(model)

print("Model prepared for compatibility")
print(f"Flask compatibility: {compatibility_info['flask']['is_compatible']}")
print(f"CoreML compatibility: {compatibility_info['coreml']['is_compatible']}")

## 5. Trace Model for Export

To ensure compatibility with various deployment targets, we'll trace the model using PyTorch's JIT.

In [ ]:
# Define input shapes
input_shapes = {
    'input_ids': [1, model_config['max_position_embeddings']],
    'attention_mask': [1, model_config['max_position_embeddings']]
}

# Trace model for export
traced_model_path = output_dir / 'traced_model.pt'
traced_model = trace_model_for_export(
    model=prepared_model,
    input_shapes=input_shapes,
    output_path=str(traced_model_path)
)

print(f"Model traced successfully and saved to {traced_model_path}")

## 6. Generate Compatibility Report

Finally, let's generate a comprehensive compatibility report for the model.

In [ ]:
# Generate compatibility report
compatibility_report = get_compatibility_report(prepared_model)

# Save report to file
report_path = output_dir / 'compatibility_report.md'
with open(report_path, 'w') as f:
    f.write(compatibility_report)

print(f"Compatibility report saved to {report_path}")
print("\nReport Preview:")
print(compatibility_report[:1000] + "...")

## 7. Summary

In this notebook, we've demonstrated how to check and ensure that your model is compatible with Flask applications and Apple's CoreML format. The key steps were:

1. Loading the model
2. Checking compatibility with Flask
3. Checking compatibility with CoreML
4. Preparing the model for compatibility
5. Tracing the model for export
6. Generating a comprehensive compatibility report

By following these steps, you can ensure that your model will work seamlessly with Flask applications and can be converted to Apple's MLmodel format for deployment on Apple devices.

## 8. Next Steps

With a compatible model, you can now proceed to:

1. **Flask Deployment**: Use the traced model in a Flask application
2. **CoreML Conversion**: Convert the traced model to CoreML format using coremltools

These steps are outside the scope of this notebook, but the model is now prepared for these deployment scenarios.